# Second student: BERT for token classification

Step 7B of the guide. BERTimbau tags each token with a BIO label, and code rebuilds the JSON
from the spans. The work is done by `bert_train.py`, the same script that runs on a laptop GPU:
align the teacher's JSON to spans, train, predict the gold test set, export to ONNX (fp32,
checked against PyTorch) and quantize the weights to int8 for the Lambda.

Run it with **Runtime → Change runtime type → T4 GPU**. It needs `raw`, `train`, `val` and
`test.jsonl` in S3, from `01_label_teacher.ipynb`.

**Credentials:** `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY` (the `pt-bio-extract-colab` user)
and `DATA_BUCKET`. In the Colab website, cell 1 reads them from **Secrets**; from VS Code or
Antigravity it asks for each one in a password box.

**If Colab disconnects**, run cell 1 again and then the training cell: the checkpoints are on
Google Drive, and training resumes from the last one.

## 1. Set up

In [ ]:
VERSION = "v1"  # the same as in the other notebooks
BERT = "neuralmind/bert-base-portuguese-cased"  # European Portuguese: "PORTULAN/albertina-100m-portuguese-ptpt-encoder"

!pip install -q -U transformers accelerate datasets onnx onnxscript "onnxruntime==1.30.0" pydantic awscli

import json, os
from google.colab import drive, userdata

from getpass import getpass

def secret(name):
    """Colab Secrets in the browser; outside it (VS Code, Antigravity) they can't be read,
    so ask for the value instead. It stays in this session's memory only."""
    if os.environ.get(name):
        return os.environ[name]
    try:
        return userdata.get(name).strip()  # a stray space in a secret breaks the AWS credentials
    except Exception:  # TimeoutException outside the browser, or a missing secret
        return getpass(f"{name}: ").strip()

for k in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "DATA_BUCKET"):
    os.environ[k] = secret(k)
os.environ["AWS_DEFAULT_REGION"] = "eu-west-1"
# Also a Python variable: shell lines below use {DATA_BUCKET}, never $DATA_BUCKET mixed with {...}
DATA_BUCKET = os.environ["DATA_BUCKET"]
S3 = f"s3://{DATA_BUCKET}/{VERSION}"

drive.mount("/content/drive")  # checkpoints go to Drive: they survive disconnects
DRIVE = f"/content/drive/MyDrive/wiki-bio-distill/{VERSION}"
RUN_DIR = f"{DRIVE}/bert"
os.makedirs(DRIVE, exist_ok=True)

url = "https://github.com/pinthoz/wiki-bio-distill.git"
if not os.path.exists("/content/repo"):
    !git clone -q {url} /content/repo
%cd /content/repo
!git pull -q
!mkdir -p data preds

!aws sts get-caller-identity --query Arn --output text   # must end in user/pt-bio-extract-colab
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
for f in ("raw", "train", "val", "test"):
    !aws s3 cp {S3}/{f}.jsonl data/ --only-show-errors
# The other students' predictions, if they exist, so the table compares everything
for f in ("base", "student", "student-q4"):
    !aws s3 cp {S3}/preds/{f}.jsonl preds/ --only-show-errors 2>/dev/null || true
!wc -l data/*.jsonl

## 2. Align, train, evaluate, export (about 10 minutes on a T4)

The script prints, in order: the share of fully aligned examples, 40 tokens with their tags,
the training, the comparison table, ten BERT errors, and the ONNX check (fp32 and three int8
variants, keeping the one whose tags agree best with PyTorch).

If `RUN_DIR` already holds a **finished** run, the script only re-evaluates it.
Add `--fresh` to train from scratch, for example after labeling again.

In [ ]:
!python bert_train.py --model {BERT} --out {RUN_DIR}

## 3. Save everything

The ONNX models go where the Lambdas read them at start-up. The predictions and the table go
next to the other students'.

In [ ]:
!aws s3 sync {RUN_DIR}/onnx/ {S3}/models/bert-token/ --delete --only-show-errors
!aws s3 ls {S3}/models/bert-token/ --human-readable
for f in ("bert", "bert-onnx", "bert-quant"):
    !aws s3 cp preds/{f}.jsonl {S3}/preds/ --only-show-errors
!aws s3 cp results.json {S3}/results-bert.json --only-show-errors
!mkdir -p {DRIVE}/preds && cp preds/bert*.jsonl {DRIVE}/preds/ && cp results.json {DRIVE}/results-bert.json

Done. The Lambdas load their model from S3 when they start, so they pick up the new one after
their next cold start. To make that happen now, from a laptop:

```bash
for f in pt-bio-extract-bert pt-bio-extract-bert-quant; do
  aws lambda update-function-configuration --function-name $f --description "model $(date +%F)" > /dev/null
done
```